## PHASE 2 — PART 2: Time Series Demand Forecasting
**Project:** Freshness Guard — AI-Driven Freshness Optimization

This section builds a complete time series forecasting pipeline:
1. **Prophet** — seasonal + holiday-aware (interpretable, explainable AI)
2. **XGBoost with Lag Features** — ML-based (highest accuracy)

Outputs feed directly into the **LangChain Demand Agent**.

### Cell 1 — Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# pip install prophet xgboost
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
print('All libraries imported successfully.')

### Cell 2 — Load & Aggregate to Daily Time Series
**Why aggregate?** Time series models need one row per day (or per day per store), not raw transaction rows.
We also fill missing dates with 0 so models see a continuous signal.

In [ ]:
df_raw = pd.read_csv('LangChain_Perfected_Freshness_Dataset.csv')
df_raw['sales_date'] = pd.to_datetime(df_raw['sales_date'])

# Aggregate: daily demand per store
daily_store = (
    df_raw
    .groupby(['sales_date', 'store_id'])
    .agg(
        sales_volume = ('sales_volume',  'sum'),
        avg_temp     = ('Temperature_C', 'mean'),
        weather_sev  = ('weather_severity_n', 'max'),
        is_holiday   = ('is_holiday', 'max'),
        is_weekend   = ('is_weekend', 'max')
    )
    .reset_index()
    .sort_values(['store_id', 'sales_date'])
)

def fill_missing_dates(df_store, date_col='sales_date'):
    """Fill date gaps so the time series has no holes."""
    full_range = pd.date_range(df_store[date_col].min(), df_store[date_col].max(), freq='D')
    df_store = df_store.set_index(date_col).reindex(full_range).reset_index()
    df_store = df_store.rename(columns={'index': date_col})
    df_store['sales_volume'] = df_store['sales_volume'].fillna(0)
    df_store[['avg_temp','weather_sev','is_holiday','is_weekend']] = (
        df_store[['avg_temp','weather_sev','is_holiday','is_weekend']].ffill().fillna(0)
    )
    return df_store

stores = daily_store['store_id'].unique()
store_dfs = {}
for store in stores:
    store_dfs[store] = fill_missing_dates(daily_store[daily_store['store_id'] == store].copy())

print(f'Stores: {len(stores)} | Date range: {daily_store["sales_date"].min().date()} to {daily_store["sales_date"].max().date()}')

### Cell 3 — Train / Test Split (Time-Based)
**Critical Rule:** NEVER use random split for time series — it leaks future data into training.
Always split by a cutoff date: train on past, test on future.

In [ ]:
DEMO_STORE = 'Store_1'
df_ts = store_dfs[DEMO_STORE].copy()

split_idx   = int(len(df_ts) * 0.80)
cutoff_date = df_ts['sales_date'].iloc[split_idx]
train_df = df_ts[df_ts['sales_date'] <  cutoff_date].copy()
test_df  = df_ts[df_ts['sales_date'] >= cutoff_date].copy()

print(f'Train: {train_df["sales_date"].min().date()} to {train_df["sales_date"].max().date()} ({len(train_df)} days)')
print(f'Test : {test_df["sales_date"].min().date()}  to {test_df["sales_date"].max().date()} ({len(test_df)} days)')

plt.figure(figsize=(14, 4))
plt.plot(train_df['sales_date'], train_df['sales_volume'], label='Train', color='steelblue')
plt.plot(test_df['sales_date'],  test_df['sales_volume'],  label='Test',  color='orange')
plt.axvline(cutoff_date, color='red', linestyle='--', label=f'Cutoff: {cutoff_date.date()}')
plt.title(f'Train / Test Split — Daily Demand ({DEMO_STORE})', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Units Sold'); plt.legend(); plt.tight_layout(); plt.show()

### Cell 4 — Model 1: Prophet
**Why Prophet?**
- Handles yearly + weekly seasonality automatically
- Built-in holiday support
- Produces confidence intervals (great for risk alerts)
- Decomposes demand into trend + seasonality (Explainable AI)
- Robust to missing data and outliers

In [ ]:
# Prophet requires columns named 'ds' (date) and 'y' (target)
train_prophet = train_df.rename(columns={'sales_date': 'ds', 'sales_volume': 'y'})
test_prophet  = test_df.rename(columns={'sales_date': 'ds', 'sales_volume': 'y'})

prophet_model = Prophet(
    yearly_seasonality      = True,
    weekly_seasonality      = True,
    daily_seasonality       = False,
    seasonality_mode        = 'multiplicative',
    changepoint_prior_scale = 0.05,
    interval_width          = 0.95
)
prophet_model.add_regressor('is_holiday')
prophet_model.add_regressor('weather_sev')
prophet_model.fit(train_prophet[['ds', 'y', 'is_holiday', 'weather_sev']])

future_df = test_df.rename(columns={'sales_date': 'ds'})[['ds', 'is_holiday', 'weather_sev']]
forecast  = prophet_model.predict(future_df)

y_true_p = test_df['sales_volume'].values
y_pred_p = np.maximum(forecast['yhat'].values, 0)

mae_p  = mean_absolute_error(y_true_p, y_pred_p)
rmse_p = np.sqrt(mean_squared_error(y_true_p, y_pred_p))
mape_p = mean_absolute_percentage_error(y_true_p + 1, y_pred_p + 1) * 100

print(f'Prophet Evaluation ({DEMO_STORE}):')
print(f'  MAE  : {mae_p:,.1f} units')
print(f'  RMSE : {rmse_p:,.1f} units')
print(f'  MAPE : {mape_p:.2f}%')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(train_df['sales_date'], train_df['sales_volume'], label='Train (Actual)', color='steelblue', alpha=0.5)
ax.plot(test_df['sales_date'],  y_true_p, label='Test (Actual)',    color='black',     linewidth=1.5)
ax.plot(test_df['sales_date'],  y_pred_p, label='Prophet Forecast', color='red',       linewidth=2)
ax.fill_between(test_df['sales_date'],
                np.maximum(forecast['yhat_lower'].values, 0),
                forecast['yhat_upper'].values,
                color='red', alpha=0.15, label='95% Confidence Interval')
ax.axvline(cutoff_date, color='gray', linestyle='--', alpha=0.7)
ax.set_title(f'Prophet Demand Forecast — {DEMO_STORE}  |  MAPE: {mape_p:.1f}%', fontsize=13)
ax.set_xlabel('Date'); ax.set_ylabel('Units Sold'); ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

### Cell 5 — Prophet Decomposition
**Why?** Decomposes demand into trend + weekly + yearly patterns.
Answers: *'Why is demand high on Saturdays?'* / *'Which month has peak demand?'*
This is the **Explainable AI** component required by the project.

In [ ]:
future_full = prophet_model.make_future_dataframe(periods=len(test_df), freq='D')
future_full['is_holiday']  = 0
future_full['weather_sev'] = 1
forecast_full = prophet_model.predict(future_full)

fig = prophet_model.plot_components(forecast_full)
fig.suptitle(f'Prophet Decomposition — {DEMO_STORE}', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()

print('Trend   -> Is overall demand growing or shrinking?')
print('Weekly  -> Which days of the week sell the most?')
print('Yearly  -> Which months / seasons have peak demand?')

### Cell 6 — Model 2: XGBoost with Lag Features
**Why XGBoost?**
- Best-in-class accuracy for tabular time series
- Uses weather, holidays, temperature as extra signals
- Handles non-linear patterns Prophet misses

**Lag features** = the model's memory:
- `lag_7d` = demand 7 days ago (captures weekly rhythm)
- `rolling_7d_mean` = smoothed demand over last 7 days

In [ ]:
def create_lag_features(df, target_col='sales_volume', lags=[1, 3, 7, 14, 30]):
    df = df.copy().sort_values('sales_date').reset_index(drop=True)
    for lag in lags:
        df[f'lag_{lag}d'] = df[target_col].shift(lag)
    df['rolling_7d_mean']  = df[target_col].shift(1).rolling(7,  min_periods=1).mean()
    df['rolling_14d_mean'] = df[target_col].shift(1).rolling(14, min_periods=1).mean()
    df['rolling_30d_mean'] = df[target_col].shift(1).rolling(30, min_periods=1).mean()
    df['rolling_7d_std']   = df[target_col].shift(1).rolling(7,  min_periods=1).std().fillna(0)
    df['day_of_week']    = df['sales_date'].dt.dayofweek
    df['day_of_month']   = df['sales_date'].dt.day
    df['month']          = df['sales_date'].dt.month
    df['week_of_year']   = df['sales_date'].dt.isocalendar().week.astype(int)
    df['quarter']        = df['sales_date'].dt.quarter
    df['is_month_end']   = df['sales_date'].dt.is_month_end.astype(int)
    df['is_month_start'] = df['sales_date'].dt.is_month_start.astype(int)
    return df

feature_cols = [
    'lag_1d', 'lag_3d', 'lag_7d', 'lag_14d', 'lag_30d',
    'rolling_7d_mean', 'rolling_14d_mean', 'rolling_30d_mean', 'rolling_7d_std',
    'day_of_week', 'day_of_month', 'month', 'week_of_year', 'quarter',
    'is_month_end', 'is_month_start',
    'avg_temp', 'weather_sev', 'is_holiday', 'is_weekend'
]

df_xgb = create_lag_features(store_dfs[DEMO_STORE].copy()).dropna()
X = df_xgb[feature_cols]
y = df_xgb['sales_volume']

split_idx_xgb = int(len(df_xgb) * 0.80)
X_train_xgb = X.iloc[:split_idx_xgb];  X_test_xgb = X.iloc[split_idx_xgb:]
y_train_xgb = y.iloc[:split_idx_xgb];  y_test_xgb = y.iloc[split_idx_xgb:]
dates_test  = df_xgb['sales_date'].iloc[split_idx_xgb:]

xgb_model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=42, n_jobs=-1
)
xgb_model.fit(X_train_xgb, y_train_xgb, eval_set=[(X_test_xgb, y_test_xgb)], verbose=False)
y_pred_xgb = np.maximum(xgb_model.predict(X_test_xgb), 0)

mae_x  = mean_absolute_error(y_test_xgb, y_pred_xgb)
rmse_x = np.sqrt(mean_squared_error(y_test_xgb, y_pred_xgb))
mape_x = mean_absolute_percentage_error(y_test_xgb + 1, y_pred_xgb + 1) * 100

print(f'XGBoost Evaluation ({DEMO_STORE}):')
print(f'  MAE  : {mae_x:,.1f} units')
print(f'  RMSE : {rmse_x:,.1f} units')
print(f'  MAPE : {mape_x:.2f}%')

plt.figure(figsize=(14, 5))
plt.plot(dates_test.values, y_test_xgb.values, label='Actual Demand',    color='black', linewidth=1.5)
plt.plot(dates_test.values, y_pred_xgb,         label='XGBoost Forecast', color='green', linewidth=2)
plt.title(f'XGBoost Demand Forecast — {DEMO_STORE}  |  MAPE: {mape_x:.1f}%', fontsize=13)
plt.xlabel('Date'); plt.ylabel('Units Sold'); plt.legend(); plt.tight_layout(); plt.show()

### Cell 7 — Feature Importance
**Why?** Shows *what* drives demand — lag history, calendar effects, or external signals.
This is the **Explainable AI** layer required by the project task.

In [ ]:
from matplotlib.patches import Patch

feat_imp_df = pd.DataFrame({
    'Feature'   : feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(15)

colors = ['#2196F3' if ('lag' in f or 'rolling' in f) else
          '#FF9800' if f in ['day_of_week','month','week_of_year','quarter',
                             'day_of_month','is_month_end','is_month_start'] else
          '#E91E63'
          for f in feat_imp_df['Feature']]

plt.figure(figsize=(9, 6))
plt.barh(feat_imp_df['Feature'], feat_imp_df['Importance'], color=colors)
plt.xlabel('Importance Score')
plt.title(f'XGBoost Feature Importance — What Drives Demand? ({DEMO_STORE})', fontsize=13)
legend_elements = [
    Patch(facecolor='#2196F3', label='Historical Demand (Lag / Rolling)'),
    Patch(facecolor='#FF9800', label='Calendar Features'),
    Patch(facecolor='#E91E63', label='External Signals (Weather / Holiday)')
]
plt.legend(handles=legend_elements, loc='lower right')
plt.tight_layout(); plt.show()

### Cell 8 — Model Comparison: Prophet vs XGBoost

In [ ]:
prophet_preds_aligned = dict(zip(test_df['sales_date'],  y_pred_p))
xgb_preds_aligned     = dict(zip(dates_test.values, y_pred_xgb))
common_dates = sorted(set(prophet_preds_aligned.keys()) & set(xgb_preds_aligned.keys()))
actual_vals  = test_df.set_index('sales_date')['sales_volume'].reindex(common_dates).values
prophet_vals = np.array([prophet_preds_aligned[d] for d in common_dates])
xgb_vals     = np.array([xgb_preds_aligned[d]     for d in common_dates])

mae_p_c  = mean_absolute_error(actual_vals, prophet_vals)
mape_p_c = mean_absolute_percentage_error(actual_vals + 1, prophet_vals + 1) * 100
mae_x_c  = mean_absolute_error(actual_vals, xgb_vals)
mape_x_c = mean_absolute_percentage_error(actual_vals + 1, xgb_vals + 1) * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for ax, preds, name, color, mae, mape in [
    (axes[0], prophet_vals, 'Prophet', 'red',   mae_p_c, mape_p_c),
    (axes[1], xgb_vals,     'XGBoost', 'green', mae_x_c, mape_x_c)
]:
    ax.plot(common_dates, actual_vals, label='Actual', color='black', linewidth=1.5)
    ax.plot(common_dates, preds, label=f'{name} Forecast', color=color, linewidth=2)
    ax.set_title(f'{name}  |  MAE: {mae:,.0f}  |  MAPE: {mape:.1f}%', fontsize=12)
    ax.set_xlabel('Date'); ax.set_ylabel('Units Sold'); ax.legend()
    ax.tick_params(axis='x', rotation=30)
plt.suptitle(f'Model Comparison: Prophet vs XGBoost — {DEMO_STORE}', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print(f'Prophet  MAE: {mae_p_c:,.1f}  MAPE: {mape_p_c:.2f}%')
print(f'XGBoost  MAE: {mae_x_c:,.1f}  MAPE: {mape_x_c:.2f}%')
winner = 'XGBoost' if mae_x_c < mae_p_c else 'Prophet'
print(f'Winner on MAE: {winner}')

### Cell 9 — 30-Day Walk-Forward Forecast (All Stores)
**Why walk-forward?** With lag features, you can't use real lag values for future dates — they don't exist yet.
So: predict day 1 → use it as lag for day 2 → predict day 2 → etc.
This is the correct multi-step forecasting approach.

Output: `demand_forecast_30days.csv` — consumed by the **LangChain Demand Agent**.

In [ ]:
FORECAST_HORIZON = 30

def forecast_next_n_days(store_df, model, feature_cols, n_days=30):
    """Walk-forward: each prediction feeds back as lag for the next step."""
    df_pred   = create_lag_features(store_df.copy()).dropna()
    last_date = df_pred['sales_date'].max()
    history   = df_pred['sales_volume'].tolist()
    future_preds, future_dates = [], []

    for i in range(1, n_days + 1):
        next_date = last_date + pd.Timedelta(days=i)
        row = {
            'lag_1d' : history[-1],
            'lag_3d' : history[-3]  if len(history) >= 3  else history[0],
            'lag_7d' : history[-7]  if len(history) >= 7  else history[0],
            'lag_14d': history[-14] if len(history) >= 14 else history[0],
            'lag_30d': history[-30] if len(history) >= 30 else history[0],
            'rolling_7d_mean' : np.mean(history[-7:]),
            'rolling_14d_mean': np.mean(history[-14:]),
            'rolling_30d_mean': np.mean(history[-30:]),
            'rolling_7d_std'  : np.std(history[-7:]),
            'day_of_week'  : next_date.dayofweek,
            'day_of_month' : next_date.day,
            'month'        : next_date.month,
            'week_of_year' : next_date.isocalendar()[1],
            'quarter'      : (next_date.month - 1) // 3 + 1,
            'is_month_end' : int(next_date == next_date + pd.offsets.MonthEnd(0)),
            'is_month_start': int(next_date.day == 1),
            'avg_temp'   : store_df['avg_temp'].median(),
            'weather_sev': 1,
            'is_holiday' : 0,
            'is_weekend' : int(next_date.dayofweek >= 5)
        }
        pred = max(0, float(model.predict(pd.DataFrame([row])[feature_cols])[0]))
        future_preds.append(pred)
        future_dates.append(next_date)
        history.append(pred)

    return pd.DataFrame({'date': future_dates, 'forecasted_demand': future_preds})

print('Generating 30-day forecasts for all stores...')
all_store_forecasts = {}
for store in stores:
    fc = forecast_next_n_days(store_dfs[store], xgb_model, feature_cols, FORECAST_HORIZON)
    fc['store_id'] = store
    all_store_forecasts[store] = fc
    print(f'  {store}: avg {fc["forecasted_demand"].mean():,.0f} units/day')

forecast_master = pd.concat(all_store_forecasts.values(), ignore_index=True)
forecast_master = forecast_master[['store_id', 'date', 'forecasted_demand']]
forecast_master['forecasted_demand'] = forecast_master['forecasted_demand'].round(0).astype(int)
forecast_master.to_csv('demand_forecast_30days.csv', index=False)
print(f'\nSaved: demand_forecast_30days.csv')
print(forecast_master.head(10))

### Cell 10 — All-Store Forecast Dashboard

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
axes = axes.flatten()
palette = plt.cm.tab10.colors

for idx, store in enumerate(sorted(stores)):
    ax   = axes[idx]
    fc   = all_store_forecasts[store]
    hist = store_dfs[store].tail(60)
    ax.plot(hist['sales_date'], hist['sales_volume'],    color='steelblue', alpha=0.6, linewidth=1.2, label='Historical')
    ax.plot(fc['date'],         fc['forecasted_demand'], color=palette[idx], linewidth=2, label='Forecast')
    ax.axvline(hist['sales_date'].max(), color='gray', linestyle='--', linewidth=0.8)
    ax.fill_between(fc['date'],
                    fc['forecasted_demand'] * 0.85,
                    fc['forecasted_demand'] * 1.15,
                    alpha=0.2, color=palette[idx])
    ax.set_title(store, fontsize=10, fontweight='bold')
    ax.set_ylabel('Units Sold', fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    ax.tick_params(axis='y', labelsize=7)

axes[0].legend(fontsize=7)
plt.suptitle('30-Day Demand Forecast — All Stores (XGBoost)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

### Cell 11 — LangChain Demand Agent Integration Layer
Converts numeric forecast into **natural-language insights** the Demand Agent uses to answer:
- *'Which store needs urgent replenishment next week?'*
- *'Is there a weekend demand surge coming?'*

In [ ]:
def generate_demand_insights(forecast_df):
    """Convert numeric forecast into natural-language context for LangChain Demand Agent."""
    insights = []
    next_week_cutoff = forecast_df['date'].min() + pd.Timedelta(days=7)
    next_week        = forecast_df[forecast_df['date'] <= next_week_cutoff]
    store_weekly     = next_week.groupby('store_id')['forecasted_demand'].sum().sort_values(ascending=False)

    insights.append('WEEKLY DEMAND FORECAST (Next 7 Days):')
    for store, demand in store_weekly.items():
        insights.append(f'  {store}: {demand:,} units forecasted')

    top_store = store_weekly.idxmax()
    low_store = store_weekly.idxmin()
    insights.append(f'\nHIGH DEMAND ALERT: {top_store} has the highest forecasted demand ({store_weekly.max():,} units). Prioritize replenishment.')
    insights.append(f'LOW DEMAND ALERT:  {low_store} has the lowest demand ({store_weekly.min():,} units). Consider redistribution or promotions.')

    forecast_df = forecast_df.copy()
    forecast_df['is_weekend_fc'] = pd.to_datetime(forecast_df['date']).dt.dayofweek >= 5
    weekend_avg = forecast_df[forecast_df['is_weekend_fc']]['forecasted_demand'].mean()
    weekday_avg = forecast_df[~forecast_df['is_weekend_fc']]['forecasted_demand'].mean()
    if weekend_avg > weekday_avg * 1.10:
        insights.append(f'\nWEEKEND SURGE: Weekend demand ({weekend_avg:,.0f}/day) is {((weekend_avg/weekday_avg)-1)*100:.0f}% higher than weekdays. Increase stock before weekends.')

    return '\n'.join(insights)

insight_text = generate_demand_insights(forecast_master)
print(insight_text)

with open('demand_agent_context.txt', 'w') as f:
    f.write(insight_text)
print('\nSaved: demand_agent_context.txt (context for LangChain Demand Agent)')
print('\nTime Series Forecasting Phase COMPLETE.')
print('Next Step: Phase 3 — Optimization (PuLP) + LangChain Agent Design')